In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json, os, sys
import pandas as pd
import datetime as dt

from rockyclickup.wrapper import Session as RCUSession
from rockyclickup.utils import response_to_dataframe as rcu_res_to_df, datetime_nearest_day
from rockyclickup.models import MODEL_LOOKUP, Client, FSA, DCA, HSA, HRA, PKG, TRN, LSA, ADO, EDU

from rockyelevate.wrapper import Session as ELVSession
from rockyelevate.utils import response_to_dataframe as elv_res_to_df

from rockydb.connection import CoreDB

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from utils.merger import Merger 


In [6]:
rcu = RCUSession()
elv = ELVSession("PROD", multithread=True, max_threads=40)
rdb = CoreDB()


In [7]:
renewals = elv.get_renewals(organization_ids=[10177])

In [8]:
renewals

[]

In [9]:
organization = elv.get_organizations(oids=[10177])

1
pool
[10177]
 - related
 - []
 - []
 - []
0 errors


In [17]:
plans = elv.get_plans_by_org(oids=[10177], detail=True)

In [24]:
plan_df = elv_res_to_df(plans)

In [34]:
for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    plan_df[col] = pd.to_datetime(plan_df[col])


display(plan_df[plan_df['plan_year.valid_from'].apply(lambda x: x.year == 2026)])

,id,parent_id,plan_year_id,prior_plan_id,is_plan,is_forfeited,is_replenished_plan,plan_code,plan_omnibus_account_id,notional_payroll_account_id,...,forfeiture_date,service_configs.DEPENDENT_CARE,rollover_date,service_configs.LIFE_STYLE,plan_account_funding_config.account_maintenance_fees_amount.account_maintenance_fees_amount,plan_account_funding_config.account_maintenance_fees_amount.account_maintenance_fees_amount_state,plan_account_funding_config.apply_account_maintenance_fees_to.apply_account_maintenance_fees_to,plan_account_funding_config.apply_account_maintenance_fees_to.apply_account_maintenance_fees_to_state,plan_account_funding_config.account_maintenance_fees_threshold.account_maintenance_fees_threshold,plan_account_funding_config.account_maintenance_fees_threshold.account_maintenance_fees_threshold_state
4,90286,50851,86659,45850.0,True,False,True,RMRFEDHRA0101202612312026,2969591,2969592,...,NaN,NaN,NaN,NaN,NaN,MODIFIABLE,NaN,MODIFIABLE,NaN,MODIFIABLE


In [26]:
plans_to_roll = plan_df[
    (plan_df['plan_year.valid_from'] == dt.datetime(2025, 1, 1)) &
    (plan_df['plan_year.valid_to'] == dt.datetime(2025, 12, 31)) &
    (~plan_df['id'].isin(plan_df['prior_plan.id'])) &
    (~plan_df['name.name'].str.startswith("zzz"))
]

plans_to_roll

,id,parent_id,plan_year_id,prior_plan_id,is_plan,is_forfeited,is_replenished_plan,plan_code,plan_omnibus_account_id,notional_payroll_account_id,...,forfeiture_date,service_configs.DEPENDENT_CARE,rollover_date,service_configs.LIFE_STYLE,plan_account_funding_config.account_maintenance_fees_amount.account_maintenance_fees_amount,plan_account_funding_config.account_maintenance_fees_amount.account_maintenance_fees_amount_state,plan_account_funding_config.apply_account_maintenance_fees_to.apply_account_maintenance_fees_to,plan_account_funding_config.apply_account_maintenance_fees_to.apply_account_maintenance_fees_to_state,plan_account_funding_config.account_maintenance_fees_threshold.account_maintenance_fees_threshold,plan_account_funding_config.account_maintenance_fees_threshold.account_maintenance_fees_threshold_state
5,45848,28622,41607,35603.0,True,False,True,RMRFEDDCA0101202512312025,1769961,1769962,...,NaN,"[{'id': 2927837, 'service': {'id': 73, 'name':...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,45849,50846,41607,35604.0,True,False,True,RMRFEDFSA0101202512312025,1769964,1769965,...,NaN,NaN,2026-01-01,"[{'id': 4196058, 'service': {'id': 844, 'name'...",NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
plans_to_roll[["name.name", "plan_code", 'account_type.account_type', 'plan_status']]

,name.name,plan_code,account_type.account_type,plan_status
5,Dependent Care FSA (DCA) 2025,RMRFEDDCA0101202512312025,DCAP,ACTIVE
6,Healthcare FSA (FSA) 2025,RMRFEDFSA0101202512312025,HCFSA,ACTIVE


In [ ]:
# org_renewals = elv.get(endpoint=f"{elv.basepath}/renewals/organization/9989")

In [21]:
py_renewals = elv.get(endpoint=f"{elv.basepath}/renewals/plan-year/41607")

In [23]:
py_renewals.json()

{'id': 11934,
 'org_id': 10177,
 'org_name': 'City of Federal Heights',
 'org_path': '\\0\\4928\\5012\\10177\\',
 'partner_name': 'Rocky Mountain Reserve',
 'plan_year_id': 41607,
 'plan_year': {'name': '01/01/2025 - 12/31/2025',
  'valid_from': '2025-01-01',
  'valid_to': '2025-12-31',
  'prior_plan_year_id': 33958},
 'status': 'RESOLVED',
 'new_plan_year': {'name': 'Plan Year 2026',
  'valid_from': '2026-01-01',
  'valid_to': '2026-12-31',
  'prior_plan_year_id': 41607},
 'plans': [{'name': 'Healthcare FSA (FSA) 2026',
   'plan_code': 'HCFSA2026',
   'parent_id': 50846,
   'account_type': 'HCFSA',
   'prior_plan_id': 45849,
   'is_custom_prefund_calc': False,
   'is_replenished': True,
   'funding_method_type': 'STANDARD_NOTIONAL',
   'is_carded': True,
   'allow_investments': False,
   'grace_period_type': 'DOES_NOT_APPLY',
   'run_out_type': 'CUSTOM',
   'is_rollover': True,
   'max_rollover_amount_type': 'IRS_LIMIT',
   'rollover_claims_type': 'ALLOW_PRIOR_YEAR_CLAIMS'},
  {'name'

In [ ]:
# plan_bodies = [p for p in plans if p.get("id") in [int(id) for id in plans_to_roll['id']]]

In [35]:
import re
from dateutil.relativedelta import relativedelta

rmrcode = "RMRFED"
organization_id = 10177

ACCOUNT_TYPE_MAP = {
    "LIFESTYLE": "LSA",
    "DCAP": "DCA",
    "HCFSA": "FSA",
    "HSA": "HSA"
}

PROD_TEMPLATE_IDS = {
    "HCFSA": 3,
    "DCAP": 4,
    "HRA": 5,
    "HSA": 6,
    "SPECIALTY": 53,
    "LIFESTYLE": 53,
    "TRANSIT": 109,
    "PARKING": 110,
    "ADOPTION": 113,
    "WELLNESS": 28974,
}


def create_plan_name(row):
    new_plan_name = row.get("name.name")
    
    number_substrings = re.findall(r'\d+', new_plan_name)
    
    for num in number_substrings:
        if int(num) > 2020:
            new_plan_name = new_plan_name.replace(num, "")
    
    new_plan_name = new_plan_name.strip()
    number_substrings = re.findall(r'\d+', new_plan_name)
    if number_substrings and new_plan_name.endswith(number_substrings[-1]):
        new_plan_name = new_plan_name.replace(number_substrings[-1], "")
    
    if "delete" in new_plan_name.lower():
        raise ValueError(f"'delete' in plan name {new_plan_name}")
    
    for template_word in ['template', 'Template', "TEMPLATE"]:
        if template_word in new_plan_name:
            new_plan_name = new_plan_name.replace(template_word, "")
    
    new_plan_name = new_plan_name.replace("  ", " ")
    
    name_split = new_plan_name.split(" ")
    name_split = [s for s in name_split if s != '']
    new_plan_name = " ".join(name_split)
    
    if row.get("account_type.account_type") == "HSA":
        return new_plan_name
    
    new_plan_name = f"{new_plan_name} 2026"
    return new_plan_name


def create_plan_code(row):
    return f"{rmrcode}{ACCOUNT_TYPE_MAP.get(row['account_type.account_type'])}{(row.get("plan_year.valid_from") + relativedelta(years = 1)).strftime("%m%d%Y")}{(row.get("plan_year.valid_to") + relativedelta(years = 1)).strftime("%m%d%Y")}"

naked_bodies = []
for index, row in plans_to_roll.iterrows():
    new_name = create_plan_name(row)
    new_plan_code = create_plan_code(row)

    new_plan = {
        "plan_year_id": 86659,
        "organization_id": organization_id,
        "parent_id": PROD_TEMPLATE_IDS.get(row['account_type.account_type']),
        "plan_code": new_plan_code,
        "name": {
            "name": new_name,
            "name_state": "MODIFIABLE"
        },
        "prior_plan_id": row.get("id"),
        "is_plan": True
    }

    naked_bodies.append(new_plan)


In [37]:
naked_bodies

[{'plan_year_id': 86659,
  'organization_id': 10177,
  'parent_id': 4,
  'plan_code': 'RMRFEDDCA0101202612312026',
  'name': {'name': 'Dependent Care FSA (DCA) 2026',
   'name_state': 'MODIFIABLE'},
  'prior_plan_id': 45848,
  'is_plan': True},
 {'plan_year_id': 86659,
  'organization_id': 10177,
  'parent_id': 3,
  'plan_code': 'RMRFEDFSA0101202612312026',
  'name': {'name': 'Healthcare FSA (FSA) 2026', 'name_state': 'MODIFIABLE'},
  'prior_plan_id': 45849,
  'is_plan': True}]

In [38]:
naked_responses = []

for i, naked_body in enumerate(naked_bodies):
    endpoint = f"{elv.basepath}/plans"
    print(endpoint)
    naked_res = elv.post(endpoint=endpoint, payload=naked_body)
    naked_responses.append(naked_res)
    print(naked_res)




https://external.prod.elevateaccounts.com/v1/plans
('https://external.prod.elevateaccounts.com/v1/plans', <Response [201]>)
https://external.prod.elevateaccounts.com/v1/plans
('https://external.prod.elevateaccounts.com/v1/plans', <Response [201]>)


In [41]:
naked_resposne_bodies = [b[1].json() for b in naked_responses]
with open("RMRFED_sent_naked_bodies.json", "w") as f:
    json.dump(naked_resposne_bodies, f, indent=4)

In [ ]:
# plan_years = elv.get_plan_years(oids=[9989])
# py_df = elv_res_to_df(plan_years)
# py_df

In [42]:
import copy


update_bodies = []
for naked_response in naked_resposne_bodies:
    prior_plan = [p for p in plans if p.get("id") == naked_response.get("prior_plan_id")][0]

    update_body = copy.deepcopy(naked_response)


    # remove fields we dont need to send back to elevate
    for key in [k for k in [
        "parent_id",
        "plan_omnibus_account_id",
        "notional_payroll_account_id",
        "notional_funding_account_id",
        "plan_year",
        "service_configs",
        "prior_plan",
    ] if k in update_body]:
        update_body.pop(key, None)

    # copy fields from prior plan
    for f in [
        "plan_primary_config",
        "plan_coverage_config",
        "plan_account_funding_config",
    ]:
        update_body[f] = prior_plan[f]

    update_body['plan_primary_config'].pop("funding_method_type")

    # fields to straight up set
    fields_to_set = {
        "plan_status": "DRAFT",
    }

    for f in fields_to_set.keys():
        update_body[f] = fields_to_set[f]

    update_bodies.append(update_body)


In [43]:
update_responses = []

for update_body in update_bodies:
    detailed_elv_res = elv.put(
        endpoint=f"{elv.basepath}/plans/{update_body['id']}",
        payload=update_body
    )
    update_responses.append(detailed_elv_res)


In [44]:
# update_responses

update_json = [u.json() for u in update_responses]

In [46]:
with open(f"RMRFED_update_responses.json", "w") as f:
    json.dump(update_json, f, indent=4)

In [ ]:
"funnding_method_type" in update_bodies[0].keys()

In [ ]:
display(list(update_bodies[0]['plan_primary_config'].keys()))

In [ ]:
update_bodies[0]['plan_primary_config']

In [ ]:
update_responses[0][1].json()